# Data collection from the primary source
### Load required libraries

In [1]:
import json
import time
import random
from datetime import datetime, timedelta
from pathlib import Path
import requests

### Configuration

In [2]:
# Define the API endpoint and polite defaults
API_URL = "https://en.wikipedia.org/w/api.php" # MediaWiki JSON API base URL
USER_AGENT = "BDACA-II-course/1.0 (educational;adashiyj@gmail.com)"
REQUEST_DELAY = 0.5  # seconds between requests
TIMEOUT = 30

# The seven most-read English Wikipedia political articles of 2025
TARGET_ARTICLES = [
    "Charlie Kirk",
    "Donald Trump",
    "Zohran Mamdani",
    "Elon Musk",
    "Jeffrey Epstein",
    "JD Vance",
    "Karoline Leavitt",
]

# Newcomer registration window
WINDOW_START = "2025-03-01T00:00:00Z"
WINDOW_END   = "2025-05-31T23:59:59Z"

# Observation windows (relative to each newcomer's first edit)
POLITENESS_WINDOW_MONTHS = 2   # collect talk page messages for first 2 months
RETENTION_WINDOW_MONTHS  = 6   # measure retention over 6 months


### Set up session and call the MediaWiki API

In [3]:
session = requests.Session() # session persists certain parameters across multiple requests (more efficient than making individual requests)
session.headers.update({"User-Agent": USER_AGENT}) # this tells Wikipedia who is making the requests

def api_get(params: dict) -> dict | None: # Returns either a dictionary or None
    """Make a safe request to the MediaWiki API with retries and rate limiting."""
    try:
        response = session.get(API_URL, params=params, timeout=TIMEOUT)
        response.raise_for_status() # checks if the HTTP request was successful 
        data = response.json()
    except requests.exceptions.RequestException as e: 
        print(f"Request error: {e}")
        return None
    except json.JSONDecodeError: 
        print("Response was not valid JSON")
        return None
    finally: # this block always executes, whether an exception occurred or not
        time.sleep(REQUEST_DELAY)

    return data

### Article revision histories

In [4]:
def get_article_revisions(title: str) -> list[dict]:
    """
    Fetch all revisions of one article made within the registration window.

    Paginates automatically until all revisions are retrieved. Each revision
    is tagged with the article title so records stay traceable after merging.

    Called for all 7 target articles before any sampling takes place.

    Parameters:
    title (str) : Wikipedia article title (e.g. "Donald Trump").

    Returns:
    list[dict]: One dict per revision, containing: revid, parentid, user, userid, timestamp, size, comment, tags, and article.
    """
    revisions = []
    params = {
        "action":  "query",
        "format":  "json",
        "prop":    "revisions",
        "titles":  title,
        "rvprop":  "ids|user|userid|timestamp|size|comment|tags",
        "rvdir":   "newer",       # oldest-first makes pagination predictable
        "rvstart": WINDOW_START,
        "rvend":   WINDOW_END,
        "rvlimit": "500",         # maximum allowed per request
    }

    while True:
        data = api_get(params)
        if data is None:
            print(f"Failed to fetch revisions for '{title}' — stopping pagination")
            break

        # Without formatversion=2, pages is a dict keyed by pageid
        pages = list(data.get("query", {}).get("pages", {}).values())
        if not pages:
            break

        for rev in pages[0].get("revisions", []):
            rev["article"] = title  # tag source article before appending
            revisions.append(rev)

        # If the API signals more pages exist, update the continuation token
        if "continue" in data:
            params["rvcontinue"] = data["continue"]["rvcontinue"]
        else:
            break  # no more pages

    print(f"  '{title}': {len(revisions)} revisions fetched")
    return revisions

### Newcomer editing histories

In [5]:
def get_user_contributions(username: str, start: str, end: str, limit: str = "500") -> list[dict]:
    """
    Fetch edits made by a user within a given time window.

    The limit parameter controls how many results are fetched per page:
        - "1"   when checking newcomer status (stop as soon as one prior edit
                is found — no need to retrieve the full history)
        - "500" when collecting the full editing history for retention

    Called only on sampled newcomers, not all candidates (data minimisation).

    Parameters:
    username : str          Wikipedia username (without 'User:' prefix).
    start    : str          ISO 8601 timestamp — start of the collection window.
    end      : str          ISO 8601 timestamp — end of the collection window.
    limit    : str          Max results per API page; default "500".

    Returns:
    list[dict]
        One dict per edit, containing: title, timestamp, revid, sizediff, tags.
    """
    contributions = []
    params = {
        "action":  "query",
        "format":  "json",
        "list":    "usercontribs",
        "ucuser":  username,
        "ucprop":  "ids|title|timestamp|sizediff|tags|comment",
        "ucdir":   "newer",
        "ucstart": start,
        "ucend":   end,
        "uclimit": limit,
    }

    while True:
        data = api_get(params)
        if data is None:
            print(f"Failed to fetch contributions for '{username}'")
            break

        contributions.extend(data.get("query", {}).get("usercontribs", []))

        # Stop paginating early when limit="1" — we only need to know
        # whether at least one prior edit exists, not retrieve all of them
        if limit == "1":
            break

        if "continue" in data:
            params["uccontinue"] = data["continue"]["uccontinue"]
        else:
            break

    return contributions


def is_newcomer(username: str, first_edit_ts: str) -> bool:
    """
    Confirm newcomer status: return True only if the user made zero edits
    before their first observed edit on a target article.

    Uses limit="1" for efficiency — the API stops after finding the first
    prior edit, so we never fetch more data than necessary.

    Parameters:
    username      : str   Wikipedia username.
    first_edit_ts : str   ISO 8601 timestamp of the first observed edit.

    Returns:
    bool
    """
    prior_edits = get_user_contributions(
        username,
        start="2001-01-01T00:00:00Z",  # Wikipedia's founding date
        end=first_edit_ts,
        limit="1",                      # stop after finding 1 prior edit
    )
    return len(prior_edits) == 0


def measure_retention(username: str, first_edit_ts: str) -> dict:
    """
    Compute the two outcome variables for a confirmed newcomer.

    Retention (Zheng et al., 2019): at least one edit made more than
    2 months after the first edit session (i.e., in months 3-6).

    Edit-session control variable: number of edits made within 60 minutes
    of the first edit (proxy for initial investment; Zheng et al., 2019).

    Called only on sampled newcomers, not all candidates (data minimisation).

    Parameters:
    username      : str   Wikipedia username.
    first_edit_ts : str   ISO 8601 timestamp of the newcomer's first edit.

    Returns:
    dict with keys:
        username        str   — Wikipedia username
        first_edit_ts   str   — timestamp of first edit
        session_edits   int   — edits within first 60 minutes
        retained        bool  — made at least 1 edit in months 3-6
        total_edits     int   — total edits in the 6-month observation window
    """
    t0 = datetime.fromisoformat(first_edit_ts.replace("Z", "+00:00"))

    # End of the 6-month observation window
    obs_end_str = (t0 + timedelta(days=RETENTION_WINDOW_MONTHS * 30)).strftime("%Y-%m-%dT%H:%M:%SZ")

    # Start of the retention window (after the 2-month politeness window)
    ret_start_str = (t0 + timedelta(days=POLITENESS_WINDOW_MONTHS * 30)).strftime("%Y-%m-%dT%H:%M:%SZ")

    # Fetch all edits over the full 6-month window (limit="500", the default)
    all_edits = get_user_contributions(username, start=first_edit_ts, end=obs_end_str)

    # Edit-session variable: edits within 60 minutes of the first edit
    session_edits = sum(
        1 for edit in all_edits
        if (datetime.fromisoformat(edit["timestamp"].replace("Z", "+00:00")) - t0)
           .total_seconds() / 60 <= 60
    )

    # Retention: at least one edit after the 2-month politeness window
    retention_edits = get_user_contributions(
        username, start=ret_start_str, end=obs_end_str
    )

    return {
        "username":      username,
        "first_edit_ts": first_edit_ts,
        "session_edits": session_edits,
        "retained":      len(retention_edits) >= 1,
        "total_edits":   len(all_edits),
    }

### Newcomer talk page messages

In [6]:
def get_talk_page_messages(username: str, first_edit_ts: str) -> list[dict]:
    """
    Retrieve all messages posted on a newcomer's user talk page within the
    first 60 days after their first edit.

    Each revision of 'User talk:<username>' that adds content (positive size
    delta) is treated as one message. Revisions with zero or negative deltas
    (blanking, archiving, bot clean-ups) are excluded.

    Called only on sampled newcomers, not all candidates (data minimisation).

    Parameters:
    username      : str   Wikipedia username.
    first_edit_ts : str   ISO 8601 timestamp of the newcomer's first edit.

    Returns:
    list[dict]
        One dict per message, containing: revid, sender (= user),
        timestamp, size, comment, tags, recipient.
    """
    t0             = datetime.fromisoformat(first_edit_ts.replace("Z", "+00:00"))
    window_end_str = (t0 + timedelta(days=POLITENESS_WINDOW_MONTHS * 30)).strftime("%Y-%m-%dT%H:%M:%SZ")

    messages = []
    params = {
        "action":  "query",
        "format":  "json",
        "prop":    "revisions",
        "titles":  f"User talk:{username}",
        "rvprop":  "ids|user|userid|timestamp|size|comment|tags",
        "rvdir":   "newer",
        "rvstart": first_edit_ts,
        "rvend":   window_end_str,
        "rvlimit": "500",
    }

    while True:
        data = api_get(params)
        if data is None:
            print(f"Failed to fetch talk page for '{username}'")
            break

        # Without formatversion=2, pages is a dict keyed by pageid
        pages = list(data.get("query", {}).get("pages", {}).values())
        if not pages:
            break

        page = pages[0]

        if page.get("missing"):   # talk page does not exist yet — no messages
            break

        for rev in page.get("revisions", []):
            if rev.get("size", 0) > 0:        # only content-adding revisions
                rev["recipient"] = username    # tag the message recipient
                messages.append(rev)

        if "continue" in data:
            params["rvcontinue"] = data["continue"]["rvcontinue"]
        else:
            break

    return messages

### Bot category lookup

In [7]:
def get_wikipedia_bot_usernames() -> set[str]:
    """
    Retrieve all usernames currently listed in 'Category:All Wikipedia bots'.

    Used as a secondary check: message senders not matched in Zheng et al.'s
    (2019) classification file but found here are labelled 'unidentified bot'
    and excluded from role-based analyses (reported separately).

    Returns:
    set[str]
        Usernames of all listed bots (without the 'User:' namespace prefix).
    """
    bot_usernames = set()
    params = {
        "action":      "query",
        "format":      "json",
        "list":        "categorymembers",
        "cmtitle":     "Category:All Wikipedia bots",
        "cmnamespace": "2",       # namespace 2 = User pages
        "cmlimit":     "500",
        "cmtype":      "page",
    }

    while True:
        data = api_get(params)
        if data is None:
            print("Failed to fetch bot category — stopping")
            break

        for member in data.get("query", {}).get("categorymembers", []):
            raw_title = member.get("title", "")
            if raw_title.startswith("User:"):
                bot_usernames.add(raw_title[len("User:"):])  # strip namespace prefix

        if "continue" in data:
            params["cmcontinue"] = data["continue"]["cmcontinue"]
        else:
            break

    print(f"  {len(bot_usernames)} bot usernames fetched")
    return bot_usernames

### Checkpoint helpers

In [8]:
def save_checkpoint(path: str | Path, records: list[dict]) -> None:
    """
    Append records to a newline-delimited JSON file (.jsonl).

    Appending rather than overwriting means progress is never lost —
    safe to call after every user so the script can resume mid-run.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_checkpoint(path: str | Path) -> list[dict]:
    """Load all records previously saved with save_checkpoint()."""
    path = Path(path)
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def get_processed_ids(path: str | Path, id_field: str) -> set:
    """Return the set of IDs already saved, so they can be skipped on resume."""
    return {r[id_field] for r in load_checkpoint(path) if id_field in r}

### Orchestration — full collection pipeline

In [9]:
def collect_all(
    output_dir: str | Path = "data/raw",
    max_newcomers: int = 10_000,
    random_seed: int = 42,
) -> None:
    """
    Run the full data collection pipeline in six steps:

        Step 1  Fetch revision histories for all 7 target articles
        Step 2  Identify + confirm newcomer candidates
        Step 3  Randomly sample up to max_newcomers
        Step 4  Measure retention  }  called only on the sample
        Step 5  Collect messages   }  → data minimisation
        Step 6  Fetch live bot category

    All outputs are written as .jsonl files to output_dir.
    Re-running safely resumes from where it stopped (checkpoint system).

    Parameters
    ----------
    output_dir    : directory to write raw data files
    max_newcomers : maximum newcomers to sample (default 10,000)
    random_seed   : seed for reproducible random sampling (default 42)
    """
    random.seed(random_seed)

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Output file paths
    revisions_path = output_dir / "revisions.jsonl"
    newcomers_path = output_dir / "newcomers.jsonl"
    retention_path = output_dir / "retention.jsonl"
    messages_path  = output_dir / "messages.jsonl"
    bots_path      = output_dir / "bot_category.jsonl"

    # ── Step 1: Revision histories ───────────────────────────────────────────
    print("\n=== Step 1: Fetching article revision histories ===")
    already_fetched = {r["article"] for r in load_checkpoint(revisions_path)}

    for title in TARGET_ARTICLES:
        if title in already_fetched:
            print(f"  Skipping '{title}' (already fetched)")
            continue
        revisions = get_article_revisions(title)
        save_checkpoint(revisions_path, revisions)

    all_revisions = load_checkpoint(revisions_path)
    print(f"Total revisions loaded: {len(all_revisions)}")

    # ── Step 2: Identify and confirm newcomers ───────────────────────────────
    print("\n=== Step 2: Identifying and confirming newcomers ===")
    already_confirmed = get_processed_ids(newcomers_path, "username")
    confirmed = list(load_checkpoint(newcomers_path))

    # Build a dict of first-seen edit per unique user across all articles
    seen: dict[str, dict] = {}
    for rev in all_revisions:
        user = rev.get("user", "")
        ts   = rev.get("timestamp", "")
        # Skip anonymous editors (no userid) and accounts flagged as bots
        if not user or rev.get("userid", 0) == 0:
            continue
        if user not in seen:
            seen[user] = {"username": user, "first_edit_ts": ts, "article": rev["article"]}

    for candidate in seen.values():
        username = candidate["username"]
        if username in already_confirmed:
            continue
        if is_newcomer(username, candidate["first_edit_ts"]):
            confirmed.append(candidate)
            save_checkpoint(newcomers_path, [candidate])
            print(f"  Confirmed newcomer: {username} ({len(confirmed)} so far)")
        # Collect a 3x buffer before sampling for a clean random draw
        if len(confirmed) >= max_newcomers * 3:
            break

    print(f"Total confirmed newcomers: {len(confirmed)}")

    # ── Step 3: Random sample ────────────────────────────────────────────────
    print(f"\n=== Step 3: Sampling up to {max_newcomers} newcomers ===")
    sampled = (
        random.sample(confirmed, max_newcomers)
        if len(confirmed) > max_newcomers
        else confirmed
    )
    print(f"Final sample: {len(sampled)} newcomers (seed={random_seed})")

    # ── Step 4: Retention measurement (sampled only) ─────────────────────────
    print("\n=== Step 4: Measuring retention ===")
    already_measured = get_processed_ids(retention_path, "username")

    for newcomer in sampled:
        username = newcomer["username"]
        if username in already_measured:
            continue
        record = measure_retention(username, newcomer["first_edit_ts"])
        save_checkpoint(retention_path, [record])
        print(f"  {username}: retained={record['retained']}, session_edits={record['session_edits']}")

    # ── Step 5: Talk page messages (sampled only) ────────────────────────────
    print("\n=== Step 5: Collecting talk page messages ===")
    already_fetched_msgs = get_processed_ids(messages_path, "recipient")

    for newcomer in sampled:
        username = newcomer["username"]
        if username in already_fetched_msgs:
            continue
        msgs = get_talk_page_messages(username, newcomer["first_edit_ts"])
        if msgs:
            save_checkpoint(messages_path, msgs)
        else:
            # Write a sentinel so we don't re-fetch this user on resume
            save_checkpoint(messages_path, [{"recipient": username, "no_messages": True}])

    # ── Step 6: Live bot category ────────────────────────────────────────────
    print("\n=== Step 6: Fetching live Wikipedia bot category ===")
    if load_checkpoint(bots_path):
        print("  Already fetched — skipping")
    else:
        bot_usernames = get_wikipedia_bot_usernames()
        save_checkpoint(bots_path, [{"username": u} for u in bot_usernames])

    print(f"\n=== Collection complete. Files written to: {output_dir.resolve()} ===")

### Entry point

In [10]:
if __name__ == "__main__":
    collect_all(
        output_dir="data/raw",
        max_newcomers=10_000,
        random_seed=42,
    )

OSError: [Errno 30] Read-only file system: 'data'

### Test if convokit works

In [ ]:
!pip3 install convokit

In [ ]:
import spacy
from convokit import Corpus, TextParser, PolitenessStrategies
from convokit.model import Utterance, Speaker

# load spacy
nlp = spacy.load("en_core_web_sm")

# initialize
ps = PolitenessStrategies()

# create a single test utterance
utt = Utterance(id="0", speaker=Speaker(id="test"), text="Could you please help me with this?")

# correct argument is spacy_nlp not spacy_parse
result = ps.transform_utterance(utt, spacy_nlp=nlp, markers=True)
print("Politeness strategies:", result.meta["politeness_strategies"])